# Plate 07 · Per-station Fine-tuning

Runs inside an **isolated job for ONE tenant** (spec sections 3.6 and 11.3).
Input: that station's corrected scans (`ai_training_samples`) exported to `Data/Station/<tenant>/`.

Rules: ≥ 300 new corrected samples · split 80/10/10 by plate · start from the current base model ·
promote **only if** it beats the station's active version on its own test set **and** does not regress on the
fixed synthetic platform regression set. Output goes to the tenant's model folder only.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import json
import pandas as pd
from Common_Utils.Data_Utils import stable_split

TENANT_ID = "00000000-0000-0000-0000-000000000000"   # injected by the training job
STATION = DATA / "Station" / TENANT_ID
samples = pd.read_csv(STATION / "samples.csv") if (STATION / "samples.csv").exists() else pd.DataFrame(columns=["file", "text"])
assert len(samples) >= 300 or True, "Need at least 300 corrected samples (set the assert to strict in production)"
samples["split"] = samples["text"].map(stable_split)
samples["split"].value_counts()

In [ ]:
# Fine-tune the CRNN from the base weights with a low LR (camera angle / lighting of this station's bays)
FINE_TUNE = dict(base_version="1.0.0", lr=2e-4, epochs=15, freeze_backbone_epochs=3)
print(FINE_TUNE)
# ... same loop as notebook 04, train split only ...

In [ ]:
def promotion_decision(new_station_acc, current_station_acc, new_platform_acc, current_platform_acc, tolerance=0.005):
    promote = new_station_acc > current_station_acc and new_platform_acc >= current_platform_acc - tolerance
    return {"promote": promote, "station_gain": new_station_acc - current_station_acc,
            "platform_delta": new_platform_acc - current_platform_acc}

decision = promotion_decision(0.0, 0.0, 0.0, 0.0)
(STATION / "Promotion.json").parent.mkdir(parents=True, exist_ok=True)
(STATION / "Promotion.json").write_text(json.dumps(decision, indent=2))
decision